# GSoC26_H — Week 1: Baselines & Ablation

**Project:** DBpedia Hindi Chapter 2026 — Fine-Tuning Indic Models for Hindi Relational Triple Extraction + HITL Feedback

**Contributor:** Nitin Singh ([@singhhnitin](https://github.com/singhhnitin))

## What this notebook does
Reproduces three baseline systems on the **full Hindi-BenchIE** dataset using the GSoC25_H evaluator (`BenchIEDetailedComparator`):

1. **IndIE** — rule-based Hindi OIE (existing extractions)
2. **GSoC25_H best system** — Gemma-3-12B + IndIE + ReAct (existing extractions)
3. **Zero-shot Gemma-3-1B** — fresh extractions on Colab T4

Outputs the **Phase 1 baseline table** (`results/baseline_table.csv`) which becomes the foundation for the full ablation table when fine-tuning is added in Phase 2.

## Runtime
~25–35 minutes on Colab T4 (most of which is the Gemma-3-1B inference pass over the full BenchIE set).

## 1. GPU & Environment Check

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Mount Google Drive & Set Project Root

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/dbpedia-hindi-gsoc'
os.makedirs(PROJECT, exist_ok=True)

# Create the folders we'll write into
for f in ['data/benchie_hindi', 'data/gsoc25h', 'src', 'results']:
    os.makedirs(f'{PROJECT}/{f}', exist_ok=True)

print('Drive mounted. Project root ready.')

## 3. Install Dependencies

Note: Gemma-3 requires `transformers>=4.50.0` for chat template support.

In [ ]:
!pip install -q \
    "transformers>=4.50.0" \
    accelerate==1.2.1 \
    bitsandbytes==0.45.0 \
    sentence-transformers==3.3.1 \
    datasets==3.2.0 \
    pandas tqdm pydantic

print('Dependencies installed.')

## 4. Clone GSoC25_H Repository

Using `--depth=1` for faster clone (full history is not needed for evaluation).

In [ ]:
gsoc_path = f'{PROJECT}/data/gsoc25h'

if not os.path.exists(f'{gsoc_path}/neural-extraction-framework'):
    os.chdir(gsoc_path)
    !git clone --depth=1 https://github.com/dbpedia/neural-extraction-framework.git
    print('Cloned.')
else:
    print('Already cloned.')

gsoc25h = f'{PROJECT}/data/gsoc25h/neural-extraction-framework/GSoC25_H'
print(f'GSoC25_H path: {gsoc25h}')

In [ ]:
# Show the GSoC25_H directory structure
for root, dirs, files in os.walk(gsoc25h):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(gsoc25h, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        print(f'{indent}  {file}')

## 5. Load the GSoC25_H BenchIE Evaluator

Reusing `BenchIEDetailedComparator` from `llm_IE/detailed_comparison_using_benchIE.py` — this is the official evaluator used in the GSoC25_H pipeline, so our numbers are directly comparable.

In [ ]:
import sys
sys.path.insert(0, f'{gsoc25h}/llm_IE')

from detailed_comparison_using_benchIE import BenchIEDetailedComparator
print('Evaluator imported successfully.')

## 6. Baseline 1: IndIE (rule-based)

Using the pre-computed extraction file shipped with GSoC25_H.

In [ ]:
import shutil, os

gold_path  = f'{gsoc25h}/IndIE/hindi-benchie/hindi_benchie_gold.txt'
indie_path = f'{gsoc25h}/IndIE/hindi-benchie/extractions/benchie_indie.txt'

results_dir = f'{PROJECT}/results'
os.makedirs(results_dir, exist_ok=True)

# Copy with the filename format the evaluator expects
shutil.copy(indie_path, f'{results_dir}/extractions_IndIE_baseline.txt')

comparator = BenchIEDetailedComparator(gold_path, results_dir)
indie_report = comparator.generate_report(
    model_name='IndIE',
    strategy='baseline',
    save_to_json=True
)

s = indie_report['overall_stats']
print(f"\nIndIE → P: {s['precision']:.4f}  |  R: {s['recall']:.4f}  |  F1: {s['f1_score']:.4f}")

## 7. Baseline 2: GSoC25_H Best System (Gemma-3-12B + IndIE + ReAct)

Using the pre-computed extractions from the best system reported in GSoC25_H.

In [ ]:
gsoc25h_extractions = (f'{gsoc25h}/IndIE/hindi-benchie/extractions/'
                       f'benchie_indie_converted_gemma3_12b_rule_react_original_prepare_for_llm.txt')

shutil.copy(gsoc25h_extractions, f'{results_dir}/extractions_GSoC25H_best.txt')

gsoc_report = comparator.generate_report(
    model_name='GSoC25H',
    strategy='best',
    save_to_json=True
)

s = gsoc_report['overall_stats']
print(f"\nGSoC25_H best → P: {s['precision']:.4f}  |  R: {s['recall']:.4f}  |  F1: {s['f1_score']:.4f}")

## 8. Baseline 3: Zero-shot Gemma-3-1B

Fresh extractions using Gemma-3-1B in zero-shot mode. Token-restricted to 80 new tokens to keep inference fast on T4.

In [ ]:
# Log in to HuggingFace (Gemma-3 requires accepting the license at huggingface.co/google/gemma-3-1b-it)
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))
print('Logged in.')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# fp16 instead of 4-bit because of bitsandbytes/transformers compatibility issues with Gemma-3.
# 1B model fits in T4 VRAM in fp16 without quantization (~2.5 GB).
# Phase 2 will use QLoRA for fine-tuning of larger variants.
tokenizer = AutoTokenizer.from_pretrained('google/gemma-3-1b-it')
model = AutoModelForCausalLM.from_pretrained(
    'google/gemma-3-1b-it',
    torch_dtype=torch.float16,
    device_map='auto'
)
print('Gemma-3-1B loaded.')

In [ ]:
import re, json
from tqdm import tqdm

# Hindi-language prompt (avoids Language-Mixing failure mode observed in pre-experiments)
PROMPT = """आप एक हिंदी भाषा विशेषज्ञ हैं।
नीचे दिए गए हिंदी वाक्य से subject, relation और object निकालें।
केवल JSON format में उत्तर दें, कुछ और नहीं:
{{"subject": "...", "relation": "...", "object": "..."}}

वाक्य: {sentence}"""

def extract_triple(sentence: str) -> dict:
    prompt = PROMPT.format(sentence=sentence)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    try:
        clean = re.sub(r'```json|```', '', response).strip()
        match = re.search(r'\{.*?\}', clean, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception:
        pass
    return {'subject': '', 'relation': '', 'object': '', 'raw': response}

# Run across the full Hindi-BenchIE sentence set
sentences = comparator.sentences
sorted_ids = sorted(sentences.keys(), key=lambda x: int(x))
print(f'Running zero-shot Gemma-3-1B on {len(sorted_ids)} sentences...')

raw_lines = []
for sent_id in tqdm(sorted_ids):
    result = extract_triple(sentences[sent_id])
    s = result.get('subject', '').strip()
    r = result.get('relation', '').strip()
    o = result.get('object',  '').strip()
    if s and r and o:
        raw_lines.append(f'{sent_id}\t{s}\t{r}\t{o}')

with open(f'{results_dir}/extractions_Gemma3_zeroshot.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(raw_lines))

print(f'Done. {len(raw_lines)} extractions saved.')

In [ ]:
gemma_report = comparator.generate_report(
    model_name='Gemma3',
    strategy='zeroshot',
    save_to_json=True
)

s = gemma_report['overall_stats']
print(f"\nZero-shot Gemma-3-1B → P: {s['precision']:.4f}  |  R: {s['recall']:.4f}  |  F1: {s['f1_score']:.4f}")

## 9. Phase 1 Deliverable: Baseline Table

Three-system comparison. Phase 2 will add a 4th column for the fine-tuned Gemma-3 + ontology alignment pipeline.

In [ ]:
import pandas as pd

rows = []
for name, report in [
    ('IndIE (rule-based)',                 indie_report),
    ('Zero-shot Gemma-3-1B',               gemma_report),
    ('GSoC25_H best (Gemma-3-12B + ReAct)', gsoc_report),
]:
    s = report['overall_stats']
    rows.append({
        'System':    name,
        'Precision': round(s['precision'], 4),
        'Recall':    round(s['recall'],    4),
        'F1':        round(s['f1_score'],  4),
        'TP':        s['total_true_positives'],
        'FP':        s['total_false_positives'],
        'FN':        s['total_false_negatives'],
    })

df = pd.DataFrame(rows)
df.to_csv(f'{PROJECT}/results/baseline_table.csv', index=False)

print('=== PHASE 1 BASELINE TABLE ===\n')
print(df.to_markdown(index=False))
print(f'\nSaved to: {PROJECT}/results/baseline_table.csv')

## 10. Next Steps

This notebook covers the aggregate P/R/F1 — Week 2 work extends it with:

1. **Per-error-type breakdown** — apply `src/evaluation/error_taxonomy.py` to each extraction file to classify failures into the 5-type taxonomy (Predicate Normalization / Language Mixing / Implicit Relation / Argument Span / Missing Triple)
2. **Ontology alignment evaluation** — apply `src/ontology/alignment.py` to the zero-shot Gemma-3 extractions; report the predicate-accuracy delta
3. **Slot accuracy split** — report subject/predicate/object accuracy separately to confirm the predicate-is-the-hard-slot hypothesis from MILIE

All three are post-processing on the extraction files saved here — no new model runs required.

Continue in `02_week2_error_analysis.ipynb`.